# 01. 수집 — 원천 raw (정규화 전)

**무엇을 하나:** 식약처(COOKRCP01) + 농정원 레시피 API의 **응답 원문 그대로**를 `data/lab/00_raw.json` 에 저장한다. 가공·정규화는 다음 노트북(02)에서.

**왜 원문을 따로 보존하나 (중요):** 원문엔 `연두부 75g(3/4모)` 처럼 분량이 들어있다. 파서(02)가 이걸 잘못 읽으면 — 예: 섹션 머리말 `고명` 을 재료로 착각 — 고쳐야 하는데, **원문이 저장돼 있으면 API 재호출 없이 02부터 다시 돌려 결과를 비교**할 수 있다. 이게 "실험 재현"의 핵심.

**파이프라인 대응:** 운영 정본 `scripts/pipeline/s0_collect_raw.py` 와 **동일 로직**. 여기선 작은 limit 데모 + `data/lab` 격리(서빙 데이터 보호).

**채널:** 식약처(~1,146건, 어디서나 호출) · 농정원(기본/재료/과정 3서비스를 `RECIPE_ID` 로 조인, ~537건).

In [ ]:
import sys,os,asyncio,json
from pathlib import Path
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B)
try: sys.stdout.reconfigure(encoding='utf-8')
except Exception: pass
from dotenv import load_dotenv; load_dotenv()
print('backend:',B,'| EMBED_PROVIDER:',os.getenv('EMBED_PROVIDER'))

In [ ]:
# s0 의 원문 수집 함수 재사용 — 정규화 전 API 원문 그대로 (파서 이원화 방지)
sys.path.insert(0, str(B/'scripts'/'pipeline'))
from s0_collect_raw import collect_cookrcp, collect_mafra

cook_rows, cook_key = asyncio.run(collect_cookrcp(20))
mafra_svc, mafra_key = asyncio.run(collect_mafra(10))   # {service: rows}, 키 없으면 {}
n_mafra = sum(len(v) for v in mafra_svc.values())
print(f'raw 수집: cookrcp {len(cook_rows)}행(key={cook_key}) | mafra {n_mafra}행(key={mafra_key})')
if not cook_rows and not mafra_svc:
    raise SystemExit('수집 0건 — FOOD_SAFETY_API_KEY / MAFRA_API_KEY 확인 후 재실행')

raw = {'cookrcp': cook_rows, 'mafra': mafra_svc}        # 채널별 원문 (s0 의 00_raw 와 동일 구조)
lab = B/'data'/'lab'; lab.mkdir(parents=True, exist_ok=True)
(lab/'00_raw.json').write_text(json.dumps(raw, ensure_ascii=False), encoding='utf-8')
print('saved -> data/lab/00_raw.json (정규화 전 원문)')
if cook_rows:   # 원문 재료텍스트 = 02 파서 입력. 정규화 전엔 이렇게 생겼다:
    print('cookrcp 원문 재료텍스트 예:', (cook_rows[0].get('RCP_PARTS_DTLS') or '')[:90])